# ENRICH-Aerial — GCP-controlled bundle adjustment

Full pipeline on the `nadir-2` sub-block: COLMAP structure from motion, then a
bundle adjustment that adds the surveyed ground control points (and optionally
the camera centres) to COLMAP's own Ceres problem.

The machinery lives in the `pycolmap_ba` package — `frontend` for the pycolmap
stages, `backend` for the extra residuals, `utils` for scoring and export — so
the cells below are the *application*: data, settings, and reading the numbers.
The maths is written up in [`docs/gcp_bundle_adjustment.md`](../docs/gcp_bundle_adjustment.md).


In [1]:
from pathlib import Path

from pycolmap import logging

from pycolmap_ba import backend, frontend, utils

import enrich_data

# --- paths ------------------------------------------------------------------
NADIR = enrich_data.DATA_ROOT / "nadir-2"
image_path = NADIR / "images"
database_path = NADIR / "colmap/database.db"
sfm_free_path = NADIR / "colmap/sparse"  # free network, as the mapper leaves it
sfm_gcp_path = NADIR / "colmap/sparse_gcp"  # after the controlled bundle

# --- what to run ------------------------------------------------------------
# Feature extraction and brute-force matching take a while on CPU, so by default
# an existing database is reused.
FORCE_OVERWRITE_DB = False
# Give the *mapper* the camera centres as pose priors. False leaves the free
# network in an arbitrary frame, which the Sim3 below then fixes.
USE_MAPPER_POSE_PRIORS = False
# Constrain the camera centres in the GCP *bundle*, like Metashape's _GT project
# (all 39 references enabled). The camera-centre accuracy row then becomes a fit
# to data the bundle was given - the honest configuration is False.
USE_CAMERA_PRIORS = True
# Let the controlled bundle free the focal length. Safe here, and only here: the
# control is what breaks the flat-block degeneracy.
REFINE_INTRINSICS = True

logging.set_log_destination(logging.INFO, Path.cwd() / "INFO.log")

## 1. Data

`load()` reads the CSVs as they are; `add_noise()` returns another object of the
same type with a simulated survey applied. `truth` is kept for scoring, `data`
is what the pipeline is allowed to see.

In [3]:
truth = enrich_data.load(subblock="nadir-2", split_control_check=True)
data = truth  # or: enrich_data.add_noise(truth) to simulate a real survey
print(enrich_data.summary(data))

# --- names used further down ------------------------------------------------
surveyed_xyz = data.marker_xyz
marker_data = data.marker_data  # columns: image_name, marker_name, u, v
camera_centers = data.camera_xyz
marker_names = data.marker_names
control_names = data.control_names
check_names = data.check_names

# --- assumed accuracies (see docs: only the ratios matter) ------------------
tie_points_sigma_px = enrich_data.tie_points_sigma_px
gcp_sigma_px = enrich_data.gcp_sigma_px
gcp_sigma_m = enrich_data.gcp_sigma_m
camera_position_sigma_m = enrich_data.camera_position_sigma_m

print(
    f"survey noise: xy {gcp_sigma_m[0] * 100:.0f} cm, "
    f"z {gcp_sigma_m[2] * 100:.0f} cm; marks {gcp_sigma_px} px"
)
print(f"GCPs: {control_names}")
print(f"CP   {check_names}")

156 marker image marks, 26 markers, 39 images
control: 18 points | check: 8 points (gcp.001, gcp.004, gcp.010, gcp.013, gcp.016, gcp.019, gcp.022, gcp.025)
survey noise: xy 2 cm, z 3 cm; marks 1.0 px
GCPs: ['gcp.002', 'gcp.003', 'gcp.005', 'gcp.006', 'gcp.007', 'gcp.008', 'gcp.009', 'gcp.011', 'gcp.012', 'gcp.014', 'gcp.015', 'gcp.017', 'gcp.018', 'gcp.020', 'gcp.021', 'gcp.023', 'gcp.024', 'gcp.026']
CP   ['gcp.001', 'gcp.004', 'gcp.010', 'gcp.013', 'gcp.016', 'gcp.019', 'gcp.022', 'gcp.025']


In [4]:
# Everything the reports need to score the block. Partial truth is fine: each
# field just enables one more row.
gt = utils.GroundTruth(
    camera_xyz=truth.camera_xyz,
    marker_xyz=truth.marker_xyz,
    control_names=truth.control_names,
    check_names=truth.check_names,
    cam_from_world=enrich_data.cam_from_world_all(truth),
    focal_px=enrich_data.FOCAL_PX,
)

## 2. Free network

Features, matches, known intrinsics, incremental mapping. Nothing here knows
about the GCPs yet.

In [ ]:
frontend.extract_and_match(
    database_path,
    image_path,
    # SIMPLE_PINHOLE has a SINGLE focal parameter. With PINHOLE, fx and fy
    # refine independently and on this flat nadir block they diverged to
    # fx=3227.6 / fy=3818.1 - an 18% anisotropic stretch.
    # That is not a similarity, so no alignment can undo it and the whole block
    # ends up ~8.5 m out.
    # The renders have no lens distortion (README), so a distortion-free model
    # with a single focal is the correct choice - and is what Metashape used.
    frontend.FeatureOptions(camera_model="SIMPLE_PINHOLE", max_num_features=8192),
    overwrite=FORCE_OVERWRITE_DB,
)


Feature extraction and matching


W20260813 18:31:47.359737 636429 feature_extraction.cc:445] Your current options use the maximum number of threads on the machine to extract features. Extracting SIFT features on the CPU can consume a lot of RAM per thread for large images. Consider reducing the maximum image size and/or the first octave or manually limit the number of extraction threads. Ignore this warning, if your machine has sufficient memory for the current settings.
I20260813 18:31:47.359898 636865 feature_extraction.cc:494] === Feature extraction ===
I20260813 18:31:47.360296 636882 sift.cc:765] Creating SIFT CPU feature extractor
I20260813 18:31:47.360349 636884 sift.cc:765] Creating SIFT CPU feature extractor
I20260813 18:31:47.360359 636883 sift.cc:765] Creating SIFT CPU feature extractor
I20260813 18:31:47.360380 636885 sift.cc:765] Creating SIFT CPU feature extractor
I20260813 18:31:47.360395 636886 sift.cc:765] Creating SIFT CPU feature extractor
I20260813 18:31:47.360435 636887 sift.cc:765] Creating SIFT 

In [7]:
# The rendered focal is known exactly, so seed it instead of estimating it.
frontend.seed_intrinsics(
    database_path,
    frontend.CameraIntrinsics(
        width=enrich_data.IMAGE_WIDTH,
        height=enrich_data.IMAGE_HEIGHT,
        params=[enrich_data.FOCAL_PX, *enrich_data.PRINCIPAL_POINT],
    ),
)


Seeding the known intrinsics
  camera 1: SIMPLE_PINHOLE [5865.1810585 3008.        2008.       ]


1

In [8]:
if USE_MAPPER_POSE_PRIORS:
    frontend.write_pose_priors(
        database_path, camera_centers, sigma_m=camera_position_sigma_m
    )

In [ ]:
rec = frontend.run_mapping(
    database_path,
    image_path,
    sfm_free_path,
    frontend.MappingOptions(
        use_prior_position=USE_MAPPER_POSE_PRIORS,
        # A flat nadir block cannot determine focal length from imagery: focal
        # trades against scene depth. Hold the intrinsics at the known values;
        # the GCP bundle can free them later, where the control breaks the
        # degeneracy.
        refine_focal_length=False,
        # COLMAP's own tie-point filter defaults, restated so the tuning surface
        # is visible. All three are NO-OPS on this block - the 80/60 overlap
        # gives strong geometry: measured triangulation angle p1 = 7.4 deg,
        # median 26.8 deg, zero points below 5 deg, max reprojection 3.98 px -
        # but they matter on weaker blocks.
        filter_max_reproj_error=4.0,
        filter_min_tri_angle_deg=1.5,
        ba_local_min_tri_angle_deg=6.0,
    ),
    seed=0,
    progress=True,
)


Incremental mapping

Images registered:   0%|                               |  0/39 [00:00<?, 0.00/s]

I20260813 18:33:05.290936 636429 incremental_pipeline.cc:348] Loading database
I20260813 18:33:05.291018 636429 database_cache.cc:73] Loading rigs...
I20260813 18:33:05.291055 636429 database_cache.cc:83]  1 in 0.000s
I20260813 18:33:05.291069 636429 database_cache.cc:91] Loading cameras...
I20260813 18:33:05.291080 636429 database_cache.cc:109]  1 in 0.000s
I20260813 18:33:05.291086 636429 database_cache.cc:117] Loading frames...
I20260813 18:33:05.291129 636429 database_cache.cc:127]  39 in 0.000s
I20260813 18:33:05.291136 636429 database_cache.cc:135] Loading matches...
I20260813 18:33:05.296514 636429 database_cache.cc:140]  441 in 0.005s
I20260813 18:33:05.296553 636429 database_cache.cc:148] Loading images...
I20260813 18:33:05.309634 636429 database_cache.cc:245]  39 in 0.013s (connected 39, loaded 39)
I20260813 18:33:05.309677 636429 database_cache.cc:259] Loading pose priors...
I20260813 18:33:05.309708 636429 database_cache.cc:267]  39 in 0.000s
I20260813 18:33:05.309715 6364

Images registered:   5%|█▍                         |  2/39 [00:00<00:08, 4.41/s]

I20260813 18:33:05.743239 636429 incremental_pipeline.cc:607] Registering image #12 (num_reg_frames=2)
I20260813 18:33:05.743276 636429 incremental_pipeline.cc:610] => Image sees 1751 / 7873 points
I20260813 18:33:05.933970 636429 incremental_pipeline.cc:87] Retriangulation and Global bundle adjustment


Images registered:   8%|██▏                        |  3/39 [00:01<00:10, 3.67/s]

...

I20260813 18:33:33.077791 636429 incremental_pipeline.cc:607] Registering image #39 (num_reg_frames=38)
I20260813 18:33:33.077826 636429 incremental_pipeline.cc:610] => Image sees 3682 / 5529 points


Images registered: 100%|███████████████████████████| 39/39 [00:28<00:00, 1.38/s]

I20260813 18:33:33.615069 636429 incremental_pipeline.cc:87] Retriangulation and Global bundle adjustment
I20260813 18:33:34.679968 636429 incremental_pipeline.cc:87] Retriangulation and Global bundle adjustment


Images registered: 100%|███████████████████████████| 39/39 [00:28<00:00, 1.38/s]Reconstruction:
	num_rigs = 1
	num_cameras = 1
	num_frames = 39
	num_reg_frames = 39
	num_images = 39
	num_points3D = 43751
	num_observations = 221673
	mean_track_length = 5.0667
	mean_observations_per_image = 5683.92
	mean_reprojection_error = 0.607413
  camera: [5865.1810585 3008.        2008.       ]


I20260813 18:33:35.781971 636429 incremental_pipeline.cc:793] Keeping successful reconstruction
I20260813 18:33:35.828254 636429 timer.cc:90] Elapsed time: 0.508 [minutes]


In [ ]:
utils.banner("Free network vs the known camera centres")
rmse = utils.camera_center_rmse(rec, camera_centers)
print(f"camera centre RMSE: {rmse:.4f} m")

# A bundle adjustment is a local optimiser: if the network sits in an arbitrary
# frame, the GCP residuals start hopeless and the solve diverges.
# A least-squares similarity over all 39 centres brings it close enough.
if rmse > 1.0:
    utils.align_to_positions(rec, camera_centers)
    print(f"after Sim3: {utils.camera_center_rmse(rec, camera_centers):.4f} m")

# A similarity absorbs uniform scale but NOT anisotropic distortion, so a large
# residual that survives the alignment means the free network is deformed -
# almost always a runaway self-calibration - and no control will rescue it.


Free network vs the known camera centres
camera centre RMSE: 325.3831 m
  Sim3 over 39 centres, scale = 27.867294
after Sim3: 0.0155 m


In [11]:
# Scored before the control enters, so the effect of the GCPs is measurable.
# No adjusted marker coordinates exist yet, hence the empty dict: only the poses
# and the intrinsics can be scored at this point.
utils.block_report(rec)
utils.accuracy_report(rec, {}, gt, "free network")


BLOCK QUALITY
  images            39 registered / 39 in block | 1 camera(s) SIMPLE_PINHOLE
  intrinsics        [5865.181 3008.    2008.   ]
  points3D          43,751      observations 221,673
  track length      mean 5.07   median 4
                    2: 1,109 (2.5%)   3-5: 30,329 (69.3%)   6-9: 8,261 (18.9%)   10+: 4,052 (9.3%)
  observations/img  min 3,415   mean 5,684   max 7,026

  tie-point reprojection error (px)
    mean 0.607   median 0.411   RMS 0.817   p90 1.455   p99 2.267   max 3.98
    > 2 px: 1,123 points (2.6%)

  redundancy        443,346 measurements - 131,490 unknowns  ->  r = 311,856

ACCURACY vs GROUND TRUTH - free network
                          bias E   bias N   bias U    RMSE 3D
                            (cm)     (cm)     (cm)       (cm)
  camera centres n=39      -0.00    +0.00    -0.00       1.55   

  camera rotations       mean 0.0000 deg   max 0.0000 deg
  focal                  5865.181  vs reference 5865.181  (+0.000)


## 3. GCP-controlled bundle adjustment

Each marker becomes a parameter block of its own, tied to the poses by its image
marks and - for the control points only - to the survey by a Gaussian prior. The
check points get rays but no prior, so their residual is the one unbiased
accuracy figure in the whole notebook.

All sigmas are given in native units (px, m); the backend expresses them in
tie-point sigma units, because COLMAP adds its tie-point residuals unweighted.

In [15]:
solverOptions = backend.SolverOptions(
    refine_focal_length=REFINE_INTRINSICS,
    # Points with fewer rays would be IGNORED by the solver, not deleted:
    # they would keep pre-bundle coordinates and still reach the export.
    # 0 = disabled; the post-solve filter below is the one that deletes.
    min_track_length=0,
    # COLMAP's global-BA default. SOFT_L1 @ 1.0 would instead down-weight the
    # reprojection tail (p95 1.77, p99 2.27, max 3.98 px) without discarding
    # redundancy - the soft alternative to hard filtering.
    loss_function_type=None,
    # COLMAP ships function_tolerance = 0, which *disables* the relative
    # stopping test and makes every solve run to the iteration cap and report
    # NO_CONVERGENCE although the cost stopped improving. 1e-8 fixes it.
    function_tolerance=1e-8,
)
markersConstraints = backend.MarkerConstraints(
    observations=marker_data,
    xyz=surveyed_xyz,
    control_names=control_names,  # the check points stay unconstrained
    sigma_px=gcp_sigma_px,
    sigma_m=gcp_sigma_m,
    robust_loss_scale=2.8,  # Cauchy on the marks, to absorb mis-identification
)
camera_posesConstraints = (
    backend.CameraPoseConstraints(
        positions=camera_centers, sigma_m=camera_position_sigma_m
    )
    if USE_CAMERA_PRIORS
    else None
)
filterOptions = backend.FilterOptions(
    max_reproj_error=4.0, min_tri_angle_deg=1.5, min_track_length=0
)

result = backend.run_bundle_adjustment(
    rec,
    solver=solverOptions,
    markers=markersConstraints,
    camera_poses=camera_posesConstraints,
    # Post-solve filtering, same thresholds as the mapper: no-ops here, kept
    # explicit so every threshold in the notebook is visible in one place.
    filters=filterOptions,
    tie_points_sigma_px=tie_points_sigma_px,
    covariance=True,
    verbose=True,
)
marker_xyz = result.marker_xyz  # adjusted marker coordinates

tie points      : sigma 0.5 px  (reference unit)
marker image obs: sigma 1.0 px -> 2.00 in tie units
marker world    : sigma [0.02 0.02 0.03] m -> [0.04 0.04 0.06] in tie units
camera positions: sigma 0.1 m -> 0.2 in tie units
gauge           : None
  problem: 221,886 blocks = 221,673 tie points + 213 added (156 marker image, 18 marker world, 39 camera pose)
  solve 1: BundleAdjustmentTerminationType.CONVERGENCE
  filtered 0 observations -> 43,751 points remain


In [18]:
cov = result.covariance
cov

In [22]:
cov.get_point_cov(1)

array([[ 8.31222560e-05, -2.79676246e-07,  7.05017320e-05],
       [-2.79676246e-07,  7.48490913e-05, -2.37906830e-06],
       [ 7.05017320e-05, -2.37906830e-06,  6.00184416e-04]])

In [14]:
utils.block_report(rec, marker_data, marker_xyz)
utils.accuracy_report(
    rec, marker_xyz, gt, "after GCP bundle", camera_priors_used=USE_CAMERA_PRIORS
)


BLOCK QUALITY
  images            39 registered / 39 in block | 1 camera(s) SIMPLE_PINHOLE
  intrinsics        [5865.136 3008.    2008.   ]
  points3D          43,751      observations 221,673
  track length      mean 5.07   median 4
                    2: 1,109 (2.5%)   3-5: 30,329 (69.3%)   6-9: 8,261 (18.9%)   10+: 4,052 (9.3%)
  observations/img  min 3,415   mean 5,684   max 7,026

  tie-point reprojection error (px)
    mean 0.607   median 0.411   RMS 0.817   p90 1.455   p99 2.267   max 3.98
    > 2 px: 1,122 points (2.6%)

  redundancy        443,346 measurements - 131,490 unknowns  ->  r = 311,856

  marker observations (156 in 39 images)
    reprojection  mean 0.07 px   median 0.06   max 0.19
    worst:  gcp.018 0.18 px,  gcp.007 0.17 px,  gcp.008 0.14 px,  gcp.023 0.14 px,  gcp.010 0.09 px

ACCURACY vs GROUND TRUTH - after GCP bundle
                          bias E   bias N   bias U    RMSE 3D
                            (cm)     (cm)     (cm)       (cm)
  camera centres n=3

## 4. Export

Binary + text model for COLMAP, plus a point cloud carrying track length and
reprojection error per point so the sparse cloud can be filtered in a viewer.

In [12]:
utils.export_reconstruction(rec, sfm_gcp_path)
utils.save_point_cloud_as_ply(rec, sfm_gcp_path / "sparse_gcp_o3d.ply")

  wrote bin + txt + ply to /home/francesco/dev/pycolmap_ba/data/ENRICH-Aerial/nadir-2/colmap/sparse_gcp


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


True